Code from README.md

In [2]:
!pip install -r requirements.txt

In [3]:
!python3 -m enformer_test

2025-12-31 02:02:12.590790: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2025-12-31 02:02:14.342030: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2025-12-31 02:02:14.408276: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-12-31 02:02:14.408339: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA RTX 4500 Ada Generation computeCapability: 8.9
coreClock: 2.58GHz coreCount: 60 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 402.38GiB/s
2025-12-31 02:02:14.408365: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2025-12-31 02:02:14.419463: I tensorflow/strea

## Running Inference  
The simplest way to perform inference is to load the model via tfhub.dev (TODO:
LINK). The input sequence length is 393,216 with the prediction corresponding to
128 base pair windows of the center 114,688 base pairs. The input sequence is
one hot encoded using the order of indices being 'ACGT' with N values being all
zeros. Note that only the central 196,608 bp of the input sequence will be used
by the Enformer model. The rest will be cropped within the model.

In [ ]:
import tensorflow as tf
import tensorflow_hub as hub

enformer = hub.load('https://tfhub.dev/deepmind/enformer/1').model

SEQ_LENGTH = 393_216

# Numpy array [batch_size, SEQ_LENGTH, 4] one hot encoded in order 'ACGT'. The
# `one_hot_encode` function is available in `enformer.py` and outputs can be
# stacked to form a batch.
inputs = tf.zeros((1, SEQ_LENGTH, 4), dtype=tf.float32)
predictions = enformer.predict_on_batch(inputs)

In [6]:
print(predictions['human'].shape)  # [batch_size, 896 (bin), 5313 (human tracks)]
print(predictions['mouse'].shape)  # [batch_size, 896 (bin), 1643 (mouse tracks)]

(1, 896, 5313)
(1, 896, 1643)


## Outputs  
For each 128 bp window, predictions are made for every track. The mapping from
track idx to track name is found in the corresponding file in the basenji
[dataset](https://github.com/calico/basenji/tree/master/manuscripts/cross2020)
folder (targets_{organism}.txt file).

As an example, to load track annotations for the human targets:

In [5]:
import pandas as pd
targets_txt = 'https://raw.githubusercontent.com/calico/basenji/0.5/manuscripts/cross2020/targets_human.txt'
df_targets = pd.read_csv(targets_txt, sep='\t')
df_targets.shape  # (5313, 8) With rows match output shape above.

(5313, 8)